# Geophysical-to-Geotechnical Inversion Toolkit

**A hybrid physics + machine learning pipeline for estimating offshore geotechnical parameters from geophysical data**

---

## Motivation

Offshore cone penetration test (CPT) and borehole campaigns for seabed characterization are expensive and slow to acquire. This is a growing bottleneck for deep-sea mineral exploration in particular, where terrain is often sparsely characterized geotechnically and near-surface sediment can be highly variable and weak.

This toolkit explores whether **geophysical data can be inverted, and subsequently used, to estimate geotechnical parameters** as a cheaper complement to direct in-situ testing — grounded in professional experience with geophysical inverse theory for seabed characterization in the offshore industry.

The toolkit chains two distinct methodologies:

1. **Physics stage** — classical surface-wave (MASW-style) dispersion-curve inversion: a synthetic layered-earth model is forward-modeled to a Rayleigh-wave dispersion curve, corrupted with realistic noise, and then inverted to recover the shear-wave velocity (Vs) profile — with explicit characterization of the inversion's non-uniqueness.
2. **ML stage** — an XGBoost-based regressor, trained on real published seismic-CPTu data, that predicts geotechnical parameters (cone resistance *qt*, sleeve friction *fs*) from Vs and depth.

These are chained in the **integration stage**: the physics stage's inverted Vs(z) profile — including its uncertainty — is fed through the ML stage's models to produce a synthetic "raw geophysics in → geotechnical log out" pipeline, with uncertainty bounds that reflect the physics inversion's non-uniqueness.

### Explicitly out of scope

- Processing real raw offshore seismic survey data end-to-end (the physics stage uses a synthetic forward model, deliberately sidestepping this)
- Full joint inversion across multiple geophysical modalities
- Any claim that this toolkit is validated or production-ready for actual offshore deployment

### A note on honesty throughout this notebook

Two different kinds of "ground truth" appear in this project, and they are not interchangeable:

- The **physics stage** is validated against a **synthetic**, hand-specified ground truth — this checks that the inversion *method* is numerically correct, not that it matches any real seabed.
- The **ML stage** is trained and tested on **real, published, measured** seismic-CPTu data — this is where the model's predictive skill is actually checked against reality.
- The **integration stage** combines a synthetic Vs profile with real-data-trained models — its output can be checked for *internal consistency* (does uncertainty propagate sensibly through the pipeline), but not for accuracy, since the synthetic profile was never paired with a real CPT.

This distinction is restated at each relevant point below.


## Environment

This notebook is designed to run inside the project's `.venv-inversion` virtual environment (Python 3.9), with `disba`, `xgboost`, `shap`, `optuna`, and `scikit-learn` installed. See `requirements.txt` at the project root.

It assumes it is being run from inside `notebooks/`, with the working directory set to the notebook's own location (VS Code's Jupyter default). If running elsewhere, adjust `PROJECT_ROOT` in the first code cell below.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Assumes this notebook runs from notebooks/, one level below the project root.
PROJECT_ROOT = Path.cwd().parent
print(f"Project root resolved to: {PROJECT_ROOT}")

# Make each src/ subpackage importable in the flat style its modules use internally.
for subdir in ["src", "src/physics", "src/ml", "src/integration"]:
    sys.path.insert(0, str(PROJECT_ROOT / subdir))

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


---
# Stage 1 — Physics: Surface-Wave Dispersion Inversion

**Goal:** define a synthetic layered-earth model with known shear-wave velocity (Vs), forward-model its Rayleigh-wave dispersion curve, corrupt it with realistic noise, then invert to recover Vs(z) — and characterize how well- or poorly-constrained that inversion is under different survey conditions.

This is a **synthetic recovery test**: standard practice in geophysics for validating an inversion method before trusting it on field data. It checks that the *method* is numerically correct; it does not, and cannot, validate against a real seabed, since the "ground truth" here is a number we invented, not something nature produced.


In [ ]:
from layered_model import LayeredEarthModel

true_model = LayeredEarthModel.example_marine_sediment_profile()

print(f"Layers (incl. half-space): {true_model.n_layers}")
print(f"Depths to top of each layer (m): {true_model.depths_m()}")
print(f"Vs profile (m/s): {true_model.vs_profile_mps}")
print(f"Vp profile (m/s, auto-estimated, Vp/Vs=2.0): {true_model.vp_profile_mps}")
print(f"Density profile (kg/m^3, auto-estimated): {true_model.density_profile_kgm3}")


The profile above represents a normally-consolidated marine sediment sequence — Vs increasing with depth from very soft near-surface mud/clay (100 m/s) to stiffer sand/clay at depth (400 m/s in the half-space) — chosen to be representative of the offshore/seabed characterization context this toolkit targets.

## Forward modeling and non-uniqueness

Two scenarios are tested, differing in how much information is available to the inversion:

- **Scenario 1** — wide frequency band (3–50 Hz), low noise (3%): representative of a well-instrumented survey with good array coverage.
- **Scenario 2** — narrow frequency band (8–25 Hz), high noise (8%): representative of a shorter/sparser real geophone array or a lower-quality survey.

For each scenario, the dispersion curve is forward-modeled, noise is added, and the inversion is run **8 times with different random optimizer starts** — the spread across these runs is the empirical signature of how non-unique (ill-posed) the inversion is under that scenario's conditions.


In [ ]:
from forward_model import forward_model_dispersion_curve, add_gaussian_noise
from inversion import invert_dispersion_curve, run_ensemble_inversion

thicknesses_m = true_model.thicknesses_m

# --- Scenario 1: wide band, low noise ---
clean_curve_s1 = forward_model_dispersion_curve(true_model)
observed_curve_s1 = add_gaussian_noise(clean_curve_s1, noise_fraction=0.03, random_seed=RANDOM_SEED)

single_result_s1 = invert_dispersion_curve(observed_curve_s1, thicknesses_m, random_seed=0)
print("Scenario 1 -- single inversion run:")
print(f"  True Vs (m/s):      {np.round(true_model.vs_profile_mps, 1)}")
print(f"  Recovered Vs (m/s): {np.round(single_result_s1.recovered_vs_mps, 1)}")
print(f"  Misfit RMSE: {single_result_s1.misfit_rmse_mps:.2f} m/s")

print("\nRunning Scenario 1 ensemble (8 runs)...")
ensemble_s1 = run_ensemble_inversion(observed_curve_s1, thicknesses_m, n_runs=8)


In [ ]:
# --- Scenario 2: narrow band, high noise ---
clean_curve_s2 = forward_model_dispersion_curve(
    true_model, freq_min_hz=8.0, freq_max_hz=25.0, n_frequencies=40
)
observed_curve_s2 = add_gaussian_noise(clean_curve_s2, noise_fraction=0.08, random_seed=RANDOM_SEED)

print("Running Scenario 2 ensemble (8 runs)...")
ensemble_s2 = run_ensemble_inversion(observed_curve_s2, thicknesses_m, n_runs=8)

ensemble_s1_vs = np.array([r.recovered_vs_mps for r in ensemble_s1])
ensemble_s2_vs = np.array([r.recovered_vs_mps for r in ensemble_s2])

print("\nEnsemble std dev per layer (m/s):")
print(f"  Scenario 1: {np.round(ensemble_s1_vs.std(axis=0), 2)}")
print(f"  Scenario 2: {np.round(ensemble_s2_vs.std(axis=0), 2)}")


## Visualizing the results


In [ ]:
from plotting import plot_vs_profile_comparison, plot_dispersion_fit, plot_ensemble_profiles

fig1 = plot_vs_profile_comparison(true_model, single_result_s1.recovered_vs_mps, show=True)


In [ ]:
fig2 = plot_dispersion_fit(
    observed_curve_s1, clean_curve_s1, single_result_s1.recovered_curve, show=True
)


In [ ]:
fig3 = plot_ensemble_profiles(
    true_model, ensemble_s1, ensemble_s2,
    scenario1_label="Scenario 1: wide band, low noise",
    scenario2_label="Scenario 2: narrow band, high noise",
    show=True,
)


## Stage 1 finding

Under **Scenario 1** (wide frequency band, low noise), the inversion recovers the true Vs profile to within ~5% per layer, essentially identically across all 8 randomized optimizer starts — a well-constrained inverse problem.

Under **Scenario 2** (narrow frequency band, high noise), the ensemble resolves into **distinct, reproducible alternate solutions** rather than random scatter — and critically, **the lowest-misfit solution is not the one closest to the true profile**. This is a clean empirical demonstration of non-uniqueness: the data alone cannot distinguish which candidate profile is correct, and naively trusting the best-fitting model would give a confidently wrong answer.

This directly motivates why the integration stage (below) propagates the *entire ensemble* forward, rather than a single "best" Vs profile.


---
# Stage 2 — Machine Learning: Vs → Geotechnical Parameters

**Goal:** predict geotechnical parameters — cone resistance (*qt*) and sleeve friction (*fs*) — from shear-wave velocity (Vs) and depth: the two quantities the physics stage's inversion actually produces.

## Data

Trained on the open dataset from Marín-Moreno et al. (2026), *"Interpretable XGBoost-based predictions of shear wave velocity from CPTu data,"* Marine Geophysical Research 47:5 — 7,475 paired CPTu–Vs measurements (after removing 10 rows with non-physical Vs=0 values) from North Sea and Austrian/German sites, with an independent 1,526-measurement test set from 45 separate SCPTu profiles.

**Note the direction:** the source paper predicts Vs *from* CPTu features (their motivating problem: CPTu is dense/cheap, Vs is sparse/expensive). This toolkit needs the **reverse** direction — Vs is what the physics stage produces; qt/fs are what we want to estimate without a real CPT. Same underlying dataset, features and targets deliberately swapped.

## Key finding: soil type is a confound

An initial attempt using Vs and depth alone performed poorly (qt R² = −0.016 — worse than predicting the mean). Diagnosis: **for the same Vs value, mean qt ranges from ~1 MPa (organic clay) to ~35 MPa (gravelly sand)** — a ~40× spread driven entirely by soil type, which Vs and depth alone cannot distinguish. Stiffness (Vs) and bearing/strength resistance (qt) are governed by different soil mechanics, and sand and clay can share similar stiffness while having very different penetration resistance.

## Five variants tested

| Variant | Configuration | qt R² | fs R² |
|---|---|---|---|
| v1 | Vs+depth only, raw target | -0.016 | 0.167 |
| v2 | Vs+depth only, log1p target (ablation) | -0.284 | 0.111 |
| v3 | log1p target + soil-type probabilities | -0.102 | 0.205 |
| v4 | raw target + soil-probs + effective stress | **0.091** | 0.155 |
| v5 | log1p target + soil-probs + effective stress | n/a | 0.192 |

The **soil-type probabilities** come from a separate XGBoost classifier trained on Vs+depth alone (~55% accuracy, 7 soil classes) — its *predicted probabilities* (not a hard label) are fed into the qt/fs models, out-of-fold for the training set to avoid leakage. The **effective overburden stress** feature (σ'v0 = assumed submerged unit weight × depth) is physically motivated by Robertson & Cabal-style Vs–CPT correlations used in practice (see Peuchen et al. 2024, offshore wind geotechnics literature).

The **log1p target transform was found to hurt more than help** (via a dedicated ablation), due to retransformation bias — converting log-space predictions back to MPa systematically underpredicts for these right-skewed targets. **qt** ended up preferring the raw target (more sensitive to this bias, given its more extreme skew); **fs** preferred log1p (smaller magnitude, more modest skew).

**Final configuration** (different recipe per target, evidence-based, not arbitrary):


In [ ]:
ml_summary_path = PROJECT_ROOT / "results" / "ml_stage_summary.txt"
print(ml_summary_path.read_text())


## Loading the trained models

Full hyperparameter search (Optuna, 30 trials × 3 models) takes roughly 15 minutes and is run via the standalone script `src/ml/train_model.py`. This notebook loads the resulting trained artifacts directly, keeping the notebook itself fast to re-run. To reproduce training from scratch:

```bash
python src/ml/train_model.py
```

Five earlier variants (the full experimentation trail summarized above) are preserved in `src/ml/experiments/`.


In [ ]:
from geotech_prediction import _load_final_models

ml_models = _load_final_models()
print("Loaded: soil-type classifier, label encoder, qt model, fs model")
print(f"Soil-type classes: {list(ml_models['soiltype_label_encoder'].classes_)}")


---
# Stage 3 — Integration: Physics-Derived Uncertainty → Geotechnical Predictions

This stage has two parts:

**A. Uncertainty propagation (synthetic).** Every member of the physics stage's Vs ensemble (both scenarios, 8 runs each) is run through the ML stage's trained models, producing a **distribution** of predicted qt(z) and fs(z) — not a single answer. The spread of that distribution is a direct, quantitative consequence of how well- or poorly-constrained the upstream physics inversion was.

**B. Accuracy check against a real measured profile (Part B, below).** The uncertainty-propagation demo above uses the physics stage's fully synthetic Vs profile, which was never paired with a real CPT — so it can show that uncertainty propagates *sensibly*, but not that predictions are *accurate*. To actually check accuracy, Part B feeds a **real, measured** SCPTu Vs(z) profile (held out from ML-stage training) through the same trained models and compares predicted qt(z)/fs(z) against the real measured values.


In [ ]:
from geotech_prediction import predict_ensemble_profiles, DEFAULT_DEPTH_GRID_M

layer_depths_top_m = true_model.depths_m()

vs_arrays_s1 = [r.recovered_vs_mps for r in ensemble_s1]
vs_arrays_s2 = [r.recovered_vs_mps for r in ensemble_s2]

print("Predicting geotechnical logs, Scenario 1 ensemble...")
qt_stack_s1, fs_stack_s1 = predict_ensemble_profiles(vs_arrays_s1, layer_depths_top_m, ml_models)

print("Predicting geotechnical logs, Scenario 2 ensemble...")
qt_stack_s2, fs_stack_s2 = predict_ensemble_profiles(vs_arrays_s2, layer_depths_top_m, ml_models)

print("Done.")


In [ ]:
fig4, axes = plt.subplots(1, 2, figsize=(12, 8), sharey=True)

for ax, s1_stack, s2_stack, label in zip(
    axes, [qt_stack_s1, fs_stack_s1], [qt_stack_s2, fs_stack_s2], ["qt (MPa)", "fs (MPa)"]
):
    for stack, color, scenario_label in [
        (s1_stack, "tab:blue", "Scenario 1 (wide band, low noise)"),
        (s2_stack, "tab:red", "Scenario 2 (narrow band, high noise)"),
    ]:
        p10 = np.percentile(stack, 10, axis=0)
        p50 = np.percentile(stack, 50, axis=0)
        p90 = np.percentile(stack, 90, axis=0)
        ax.plot(p50, DEFAULT_DEPTH_GRID_M, color=color, linewidth=2, label=f"{scenario_label} (median)")
        ax.fill_betweenx(DEFAULT_DEPTH_GRID_M, p10, p90, color=color, alpha=0.2,
                          label=f"{scenario_label} (10th-90th pctile)")
    ax.set_xlabel(label)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7, loc="lower right")

axes[0].set_ylabel("Depth (m)")
axes[0].invert_yaxis()
fig4.suptitle(
    "Synthetic geotechnical log: predicted qt(z), fs(z) with uncertainty\n"
    "(band width reflects physics-stage survey quality, not real measurement error)"
)
fig4.tight_layout()
plt.show()


## Integration finding


In [ ]:
integration_summary_path = PROJECT_ROOT / "results" / "integration_summary.txt"
print(integration_summary_path.read_text())


## Part B — Case study: validating against a real measured profile

The uncertainty-propagation demo above (Part A) uses the physics stage's synthetic Vs profile — useful for showing *how* uncertainty propagates, but with no real measurement to check accuracy against. Here, a **real, measured** SCPTu profile from the ML stage's held-out test set is used instead.

**Profile chosen: HKW_SCPT02** — one of the 45 held-out test profiles (52.5 m depth coverage, 43 matched Vs points, Vs range 143–447 m/s). This is also the exact profile the source paper's own demonstration script (`run_CPTu_to_Vs.py`) uses as its default example.

Because this Vs(z) is real and continuous (not the physics stage's piecewise-constant layered output), it also sidesteps the local prediction instability seen when evaluating the models on synthetic, constant-Vs-within-a-layer input (see note at the end of Part A's figure, if oscillation was visible there).


In [ ]:
from load_data import load_testing_data
from geotech_prediction import predict_qt_fs_from_vs_depth

test_df = load_testing_data()
case_study_df = (
    test_df[test_df["source_sheet"] == "HKW_SCPT02"]
    .sort_values("depth_m")
    .reset_index(drop=True)
)
print(f"Case study profile: HKW_SCPT02, {len(case_study_df)} points, "
      f"depth range {case_study_df['depth_m'].min():.1f}-{case_study_df['depth_m'].max():.1f} m")

qt_pred_case, fs_pred_case = predict_qt_fs_from_vs_depth(
    case_study_df["vs_mps"].values, case_study_df["depth_m"].values, ml_models
)
case_study_df["qt_mpa_pred"] = qt_pred_case
case_study_df["fs_mpa_pred"] = fs_pred_case


In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

qt_mae_case = mean_absolute_error(case_study_df["qt_mpa"], case_study_df["qt_mpa_pred"])
qt_r2_case = r2_score(case_study_df["qt_mpa"], case_study_df["qt_mpa_pred"])
fs_mae_case = mean_absolute_error(case_study_df["fs_mpa"], case_study_df["fs_mpa_pred"])
fs_r2_case = r2_score(case_study_df["fs_mpa"], case_study_df["fs_mpa_pred"])

print(f"HKW_SCPT02 -- qt: MAE={qt_mae_case:.2f} MPa, R^2={qt_r2_case:.3f}")
print(f"HKW_SCPT02 -- fs: MAE={fs_mae_case:.3f} MPa, R^2={fs_r2_case:.3f}")
print(f"\n(For context, full test-set (1,526 points) performance: "
      f"qt R^2=0.091, fs R^2=0.205 -- see results/ml_stage_summary.txt)")


In [ ]:
print(f"Measured qt range: {case_study_df['qt_mpa'].min():.1f} - {case_study_df['qt_mpa'].max():.1f} MPa")
print(f"Predicted qt range: {case_study_df['qt_mpa_pred'].min():.1f} - {case_study_df['qt_mpa_pred'].max():.1f} MPa")


**A specific, confirmed limitation:** predicted qt tops out around 31 MPa while measured qt reaches 50.5 MPa — predicted maximum lands almost exactly at the *training data's* 75th percentile (~32 MPa). This is not a bug; it's a well-known structural property of tree-based models like XGBoost: predictions are averages over training-data leaf values, so they **cannot extrapolate beyond the range of target values seen in training** for similar input combinations — unlike a linear model, which could (rightly or wrongly) project beyond the training range. High-qt combinations of Vs, depth, and soil type were comparatively rare in the training data, so the model has no basis to predict values that high, regardless of how high the real qt goes at this specific site.

**On the R² comparison:** this single profile's qt R² (0.284) exceeds the full test-set aggregate (0.091) — this is not a contradiction. R² measures variance explained *relative to that dataset's own variance*. This profile has a wide qt range and the model tracks its overall trend reasonably well relative to that scale, even while individual point-to-point predictions are noisy. The aggregate R² pools 45 different sites with different depositional histories and qt ranges; a model can track within-site trends reasonably while still struggling with between-site differences — consistent with the soil-type/site-generalization limitation diagnosed earlier in Stage 2. The MAE (10.53 MPa, against this profile's ~48 MPa measured range) is the more sober number to keep in view: a genuine partial success, not a strong one.


In [ ]:
fig5, axes = plt.subplots(1, 2, figsize=(10, 8), sharey=True)

axes[0].plot(case_study_df["qt_mpa"], case_study_df["depth_m"], "o-", color="black", label="Measured", markersize=4)
axes[0].plot(case_study_df["qt_mpa_pred"], case_study_df["depth_m"], "o-", color="tab:orange", label="Predicted", markersize=4)
axes[0].set_xlabel("qt (MPa)")
axes[0].set_ylabel("Depth (m)")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(case_study_df["fs_mpa"], case_study_df["depth_m"], "o-", color="black", label="Measured", markersize=4)
axes[1].plot(case_study_df["fs_mpa_pred"], case_study_df["depth_m"], "o-", color="tab:orange", label="Predicted", markersize=4)
axes[1].set_xlabel("fs (MPa)")
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[0].invert_yaxis()
fig5.suptitle("Case study: HKW_SCPT02 -- predicted vs. measured (real data, held-out test profile)")
fig5.tight_layout()

out_path = PROJECT_ROOT / "results" / "figures" / "case_study_HKW_SCPT02_predicted_vs_measured.png"
fig5.savefig(out_path, dpi=150)
print(f"Saved: {out_path}")
plt.show()


**Reading this figure:** the predicted curves are naturally smooth (real Vs profiles vary continuously, unlike the physics stage's piecewise-constant synthetic input), and can be compared directly against genuine measurements — this is the one figure in this notebook checking real predictive accuracy, as distinct from internal-consistency or uncertainty-propagation checks elsewhere.

The MAE/R² for this single profile will differ from the aggregate test-set numbers (a single site's soil conditions and depth range are a much smaller, more specific sample than the full 1,526-point test set) — both numbers are meaningful, but for different purposes: the aggregate number characterizes overall model skill; this single-site number shows what a user would actually see applying the model to one real location.


---
# Conclusions

## What this toolkit demonstrates

1. **The physics-stage inversion pipeline is numerically correct** — validated via synthetic recovery test, recovering a known Vs profile to within ~5% under favorable survey conditions.
2. **Non-uniqueness in surface-wave inversion is real, quantifiable, and reproducible** — under degraded survey conditions, the inversion resolves into distinct alternate solutions, and the best-fitting solution is not necessarily the true one.
3. **Predicting geotechnical parameters from Vs alone has a genuine, physically-grounded ceiling** — soil-type ambiguity (the same Vs corresponding to very different qt depending on lithology) limits achievable accuracy, and this ceiling was diagnosed quantitatively, not just observed.
4. **Uncertainty propagates end-to-end through the full pipeline** — a degraded physics-stage survey visibly and quantitatively widens the final geotechnical prediction's confidence band, directly linking upstream survey design decisions to downstream prediction trust.
5. **The trained models produce genuinely reasonable predictions on a real, held-out measured profile** (Stage 3, Part B) — providing an actual accuracy check against real data, distinct from the synthetic internal-consistency demonstration in Part A.

## Relevance to offshore and deep-sea-mining site characterization

Offshore and deep-sea-mining site investigations face a persistent trade-off between geotechnical data density (expensive, slow to acquire) and geophysical data density (comparatively cheap, spatially extensive). This toolkit's core finding — that Vs-only geotechnical prediction is fundamentally limited by soil-type ambiguity, and that this limitation is quantifiable rather than just anecdotal — has a direct practical implication: **a purely geophysics-driven characterization workflow should be paired with even sparse soil-type constraints** (e.g. a handful of boreholes, or Vp/Vs-ratio-based lithology discrimination, per Masri & Takács 2023) rather than attempting geotechnical prediction from a single elastic parameter alone. This mirrors established practice in hydrocarbon and geothermal seismic lithology discrimination, extended here to the offshore geotechnical domain.

## Limitations (stated explicitly)

- The physics stage's synthetic validation checks the inversion *method*, not agreement with any real seabed.
- Layer thicknesses are treated as known/fixed in the inversion (a standard simplifying assumption, not jointly inverted with velocity).
- Vp/Vs ratio (2.0) and the Vs–density relation are fixed empirical assumptions, not independently constrained.
- The ML stage's soil-type classifier achieves only ~55% accuracy from Vs+depth alone — a real, quantified limitation, not hidden.
- The effective overburden stress feature uses a constant assumed submerged unit weight (9 kN/m³), not site-specific data.
- Fundamental-mode-only surface-wave inversion is assumed; real field data can contain higher-mode energy.

## References

- Marín-Moreno, H. et al. (2026). Interpretable XGBoost-based predictions of shear wave velocity from CPTu data. *Marine Geophysical Research*, 47:5.
- Masri, E. N., & Takács, E. (2023). Simultaneous model-based inversion of pre-stack 3D seismic data targeting a deep geothermal reservoir, Northwest Hungary. *Acta Geodaetica et Geophysica*.
- Peuchen, J. et al. (2024). Small strain shear modulus derived from offshore seismic reflection data. *Proc. 7th Int. Conf. on Geotechnical and Geophysical Site Characterization*, Barcelona.
- Adler, A., Araya-Polo, M., & Poggio, T. (2021). Deep learning for seismic inverse problems: toward the acceleration of geophysical analysis workflows. *IEEE Signal Processing Magazine*, 38(2).
